# NEM Negative Price Forecasting: Data Extraction

This notebook downloads historical 5-minute dispatch data for South Australia (SA1) from AEMO's NEMWeb portal using [NEMOSIS](https://github.com/UNSW-CEEM/NEMOSIS) (National Electricity Market Open Source Information System), a Python package developed by UNSW.

The extracted data: spot prices and regional dispatch summaries — is saved locally as CSV files for use in downstream modelling.

## 1. Install Dependencies

Install `nemosis` and supporting libraries. NEMOSIS downloads CSV data from AEMO's NEMWeb portal and caches it as Feather files for faster subsequent access.

In [ ]:
!pip install nemosis
import sys
print(sys.executable)

## 2. Verify Python Environment

Confirm the Python interpreter being used. Gave me headaches before. 

In [ ]:
import sys
print(sys.executable)

### Install via subprocess (alternative) might not need to run


In [ ]:
import subprocess
subprocess.check_call(['/usr/local/bin/python3', '-m', 'pip', 'install', 'nemosis', 'pandas', 'numpy', 'matplotlib', 'scikit-learn'])

## 3. Configure Data Extraction

Set the time window and local cache directory for NEMOSIS.

- `start_time` / `end_time`: Date range for the query (full calendar year 2024)
- `cache`: Directory where NEMOSIS stores downloaded and cached Feather files

> **Note:** NEMOSIS caches raw CSV downloads as Feather files in `cache`. Subsequent calls reuse this cache, so re-running is much faster after the first download.

In [ ]:
import os
import pandas as pd
from nemosis import dynamic_data_compiler

os.makedirs('../data', exist_ok=True)

start_time = '2023/01/01 00:00:00'
end_time = '2024/12/31 00:00:00'
cache = '../data'

## 4. Fetch Spot Prices — `DISPATCHPRICE`

Downloads 5-minute dispatch pricing data for SA1 from the `DISPATCHPRICE` AEMO table.

| Column | Description |
|--------|-------------|
| `SETTLEMENTDATE` | Datetime at the **end** of the 5-minute dispatch interval (AEST) |
| `REGIONID` | NEM region identifier — filtered to `SA1` (South Australia) |
| `RRP` | Regional Reference Price ($/MWh) — the spot price for energy in the region after market adjustments |


In [ ]:
prices = dynamic_data_compiler(
    start_time, end_time,
    'DISPATCHPRICE',
    cache,
    select_columns=['SETTLEMENTDATE', 'REGIONID', 'RRP'],
    filter_cols=['REGIONID'],
    filter_values=(['SA1'],)
)

## 5. Fetch Regional Dispatch Summary: `DISPATCHREGIONSUM`

Downloads 5-minute regional dispatch summary data for SA1 from the `DISPATCHREGIONSUM` AEMO table.

| Column | Description |
|--------|-------------|
| `SETTLEMENTDATE` | Datetime at the end of the 5-minute dispatch interval (AEST) |
| `REGIONID` | NEM region identifier — filtered to `SA1` |
| `TOTALDEMAND` | Regional demand in MW, **excluding** load served by scheduled dispatch units |

In [ ]:
regionsum = dynamic_data_compiler(
    start_time, end_time,
    'DISPATCHREGIONSUM',
    cache,
    select_columns=['SETTLEMENTDATE', 'REGIONID', 'TOTALDEMAND'],

    filter_cols=['REGIONID'],
    filter_values=(['SA1'],)
)

## 6. Export Data

Save the two datasets as CSV files to the local `../data/` directory for use in downstream feature engineering and modelling.

In [ ]:
prices.to_csv('../data/sa1_prices_2024.csv', index=False)
regionsum.to_csv('../data/sa1_regionsum_2024.csv', index=False)

print("Done. Files saved to Local.")